# NLP Assignment Presentation Notebook

This notebook follows the assignment requirement exactly: it uses the full GPAC corpus, not a sample, and runs Question 1.1 to Question 1.8 before the intrinsic and extrinsic evaluation sections.

- Question 1: full-corpus preprocessing and n-gram analysis
- Question 2: intrinsic evaluation
- Question 3: extrinsic evaluation
- Question 4: predictive text system


In [3]:
import os
import sys
import random
import math
import re
from collections import Counter

repo_dir = os.getcwd()
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

from ngram_model import NGramLanguageModel
from q4_predictive_system import AmharicPredictiveSystem

CORPUS_PATH = '/home/genesis/Downloads/GPAC.txt'
print(f'Repository root: {repo_dir}')
print('Imports ready.')

Repository root: /home/genesis/Desktop/NLP_assignment_presentation/NLP_AssignmentPRESENTATION
Imports ready.


In [ ]:
# Full-corpus preprocessing specification for the assignment
SENTENCE_SPLIT = re.compile(r'(?:፡፡)+|።+|[!?]+')
PUNCTUATION_RE = re.compile(r"[፣፤።፡፨!?.,;:\-/\[\]\(\){}\"“”']+")

repl = {
    # ሀ group (ሀ, ሐ, ኀ -> ሀ)
    'ሐ': 'ሀ', 'ሑ': 'ሁ', 'ሒ': 'ሂ', 'ሓ': 'ሃ', 'ሔ': 'ሄ', 'ሕ': 'ህ', 'ሖ': 'ሆ',
    'ኀ': 'ሀ', 'ኁ': 'ሁ', 'ኂ': 'ሂ', 'ኃ': 'ሃ', 'ኄ': 'ሄ', 'ኅ': 'ህ', 'ኆ': 'ሆ',

    # ሰ group (ሠ -> ሰ)
    'ሠ': 'ሰ', 'ሡ': 'ሱ', 'ሢ': 'ሲ', 'ሣ': 'ሳ', 'ሤ': 'ሴ', 'ሥ': 'ስ', 'ሦ': 'ሶ',

    # አ group (ዐ -> አ)
    'ዐ': 'አ', 'ዑ': 'ኡ', 'ዒ': 'ኢ', 'ዓ': 'ኣ', 'ዔ': 'ኤ', 'ዕ': 'እ', 'ዖ': 'ኦ',

    # ጸ group (ፀ -> ጸ)
    'ፀ': 'ጸ', 'ፁ': 'ጹ', 'ፂ': 'ጺ', 'ፃ': 'ጻ', 'ፄ': 'ጼ', 'ፅ': 'ጽ', 'ፆ': 'ጾ'
}

PROTECTED_ABBREVIATIONS = [
    'ዶ/ር',
    'ኣ.ም',
    'እ.ኤ.አ',
    'ት/ቤት',
    'ወ/ሮ',
    'መ/ቤት',
    'ጠ/ሚ'
]
PROTECTED_ABBREVIATIONS_SORTED = sorted(PROTECTED_ABBREVIATIONS, key=len, reverse=True)

ABBREVIATION_PATTERNS = [
    (r'ዶ\s*/\s*ር', 'ዶ/ር'),
    (r'ኣ\s*\.?\s*ም', 'ኣ.ም'),
    (r'እ\s*\.\s*ኤ\s*\.\s*አ', 'እ.ኤ.አ'),
    (r'እ\s*ኤ\s*አ', 'እ.ኤ.አ'),
    (r'ት\s*/\s*ቤት', 'ት/ቤት'),
    (r'ወ\s*/\s*ሮ', 'ወ/ሮ'),
    (r'መ\s*/\s*ቤት', 'መ/ቤት'),
    (r'ጠ\s*/\s*ሚ', 'ጠ/ሚ'),
]


def protect_abbreviations(text):
    safe_text = text
    for abbr in PROTECTED_ABBREVIATIONS_SORTED:
        placeholder = f'__ABBR_{abbr.replace("/", "_").replace(".", "_")}__'
        safe_text = safe_text.replace(abbr, placeholder)

    # remove punctuation but keep these contracted forms intact
    cleaned = PUNCTUATION_RE.sub(' ', safe_text)
    for abbr in PROTECTED_ABBREVIATIONS_SORTED:
        placeholder = f'__ABBR_{abbr.replace("/", "_").replace(".", "_")}__'
        cleaned = cleaned.replace(placeholder, abbr)
    return ' '.join(cleaned.split())


def normalize_for_assignment(text):
    text = text.replace('፡', ' ')
    for src, dst in repl.items():
        text = text.replace(src, dst)
    for pattern, replacement in ABBREVIATION_PATTERNS:
        text = re.sub(pattern, replacement, text)
    text = protect_abbreviations(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize_for_assignment(sentence):
    normalized = normalize_for_assignment(sentence)
    tokens = []
    for token in normalized.split():
        if not token:
            continue
        if re.fullmatch(r'\d+', token):
            continue
        if re.search(r'[\u1200-\u137F]', token):
            tokens.append(token)
    return tokens


examples = [
    'ኢትዮጵያ በአፍሪካ ትገኛለች።',
    'ዶ/ር አበበ በቤት እንደሚመጣ ነው፡፡ የወቅቱ ምርት ጠቃሚ ነው።',
    'ኣ ም ዓም እ ኤ አ ፈረንሳይ እንደገና ተመልከተ፡፡',
    'ዶ / ር እ.ኤ.አ ወ / ሮ በት / ቤት ውስጥ ነበሩ።'
]

for ex in examples:
    print('ORIGINAL:', ex)
    print('TOKENS  :', tokenize_for_assignment(ex))
    print('---')

print('Assignment preprocessing configuration loaded.')

SyntaxError: invalid character '፳' (U+1373) (823838038.py, line 90)

## Question 1.1 — Full-corpus sentence and vocabulary statistics

This cell counts all sentences in the full GPAC corpus using Amharic sentence delimiters and then builds the vocabulary after normalization, number filtering, punctuation removal, and abbreviation protection.

In [53]:
# 1.1 Full-corpus statistics using the assignment preprocessing rules
raw_sentences = []
tokenized_sentences = []

with open(CORPUS_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    corpus_text = f.read()

for segment in SENTENCE_SPLIT.split(corpus_text):
    sentence = segment.strip()
    if not sentence:
        continue
    raw_sentences.append(sentence)
    tokens = tokenize_for_assignment(sentence)
    if tokens:
        tokenized_sentences.append(tokens)

all_tokens = [tok for sent in tokenized_sentences for tok in sent]
word_counter = Counter(all_tokens)
hapax_count = sum(1 for count in word_counter.values() if count == 1)
hapax_percentage = (hapax_count / len(word_counter)) * 100 if word_counter else 0.0

print(f'Total raw sentences in corpus: {len(raw_sentences)}')
print(f'Total tokenized sentences: {len(tokenized_sentences)}')
print(f'Total tokens: {len(all_tokens)}')
print(f'Vocabulary size: {len(word_counter)}')
print(f'Hapax Legomena count: {hapax_count}')
print(f'Hapax Legomena percentage: {hapax_percentage:.2f}%')
print('First sentence:')
print(raw_sentences[0])
print('First tokenized sentence:')
print(tokenized_sentences[0])
print('Top 10 words:')
print(word_counter.most_common(10))

Total raw sentences in corpus: 5059044
Total tokenized sentences: 5056326
Total tokens: 76103547
Vocabulary size: 1874179
Hapax Legomena count: 778309
Hapax Legomena percentage: 41.53%
First sentence:
ምን መሰላችሁ
First tokenized sentence:
['ምን', 'መሰላችሁ']
Top 10 words:
[('ነው', 1165381), ('ላይ', 754500), ('ውስጥ', 346488), ('ወደ', 330957), ('እና', 318774), ('ጋር', 291509), ('ነበር', 274619), ('ግን', 271410), ('ጊዜ', 243955), ('ነገር', 217616)]


## Question 1.2 — N-gram construction from the full corpus

This cell extracts all n-grams for n = 1, 2, 3, and 4 from the cleaned corpus. Punctuation markers are removed before counting, and the most frequent sequences are printed.

In [54]:
# 1.2 Full-corpus n-gram counts for n = 1, 2, 3, and 4

def extract_ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

ngram_counters = {n: Counter() for n in [1, 2, 3, 4]}
for sent in tokenized_sentences:
    for n in [1, 2, 3, 4]:
        for gram in extract_ngrams(sent, n):
            ngram_counters[n][gram] += 1

for n in [1, 2, 3, 4]:
    total_count = sum(ngram_counters[n].values())
    print(f'\n=== Top 10 {n}-grams (markers excluded) ===')
    for idx, (gram, count) in enumerate(ngram_counters[n].most_common(10), 1):
        prob = count / total_count if total_count else 0.0
        print(f'{idx:>2}. {" ".join(gram):<30} count={count:<8} P={prob:.6f}')


=== Top 10 1-grams (markers excluded) ===
 1. ነው                             count=1165381  P=0.015313
 2. ላይ                             count=754500   P=0.009914
 3. ውስጥ                            count=346488   P=0.004553
 4. ወደ                             count=330957   P=0.004349
 5. እና                             count=318774   P=0.004189
 6. ጋር                             count=291509   P=0.003830
 7. ነበር                            count=274619   P=0.003608
 8. ግን                             count=271410   P=0.003566
 9. ጊዜ                             count=243955   P=0.003206
10. ነገር                            count=217616   P=0.002859

=== Top 10 2-grams (markers excluded) ===
 1. ነገር ግን                         count=44726    P=0.000630
 2. ብቻ ሳይሆን                        count=32876    P=0.000463
 3. ማለት ነው                         count=32380    P=0.000456
 4. ቀን ኣ.ም                         count=32192    P=0.000453
 5. አዲስ አበባ                        count=31356    P=0.000441

## Question 1.3 — N-gram probabilities

This cell trains a model for each n and prints the strongest n-grams together with their probabilities.

In [55]:
# 1.3 Top n-grams and their probabilities
models = {}
for n in [1, 2, 3, 4]:
    model = NGramLanguageModel(n=n, use_padding=False)
    model.train(tokenized_sentences)
    models[n] = model

for n in [1, 2, 3, 4]:
    model = models[n]
    top_ngrams = sorted(model.n_gram_counts.items(), key=lambda item: (-item[1], item[0]))[:10]
    print(f'\nTop 10 {n}-grams:')
    for rank, (gram, count) in enumerate(top_ngrams, 1):
        joint_p = count / model.total_ngrams if model.total_ngrams else 0.0
        cond_p = model.mle_probability(gram[-1], gram[:-1])
        print(f'{rank}. {gram} | count={count} | joint={joint_p:.6f} | cond={cond_p:.6f}')

: 

## Question 1.4 — Conditional probability

This cell evaluates conditional probability of a word given the previous context.

In [ ]:
# 1.4 Conditional probability P(word | previous word)
bigram_model = models[2]
example_pairs = [
    ('ኢትዮጵያ', 'ውስጥ'),
    ('ኢትዮጵያ', 'በአፍሪካ'),
    ('አዲስ', 'አበባ'),
    ('የቡና', 'ምርት'),
    ('የቡና', 'ቤት')
]

for prev, curr in example_pairs:
    prev_count = bigram_model.unigram_counts.get(prev, 0)
    pair_count = bigram_model.n_gram_counts.get((prev, curr), 0)
    p = (pair_count / prev_count) if prev_count else 0.0
    print(f'{prev} -> {curr}: pair_count={pair_count}, prev_count={prev_count}, P={p:.6f}')

## Question 1.5 — Stopword removal on the full corpus

This cell removes high-frequency Amharic stopwords from every sentence, then compares the vocabulary before and after filtering.

In [ ]:
# 1.5 Remove stopwords from the full corpus
# The preprocessing is retained from Q1 and applied to the full corpus without sampling.
from amharic_preprocessor import remove_stopwords

filtered_sentences = [remove_stopwords(sent) for sent in tokenized_sentences]
filtered_sentences = [sent for sent in filtered_sentences if sent]

print(f'Original sentences: {len(tokenized_sentences)}')
print(f'Sentences after stopword removal: {len(filtered_sentences)}')
print('Before stopword removal:')
print(tokenized_sentences[10])
print('After stopword removal:')
print(filtered_sentences[10])

filtered_tokens = [tok for sent in filtered_sentences for tok in sent]
filtered_counter = Counter(filtered_tokens)
print(f'Total filtered tokens: {len(filtered_tokens)}')
print(f'Filtered vocabulary size: {len(filtered_counter)}')
print('Most common filtered words:')
print(filtered_counter.most_common(10))

## Question 1.6 — Word cloud generation

This cell produces word-cloud visualizations before and after stopword removal from the full corpus.

In [ ]:
# 1.6 Create word clouds before and after stopword removal from the full corpus
from wordcloud import WordCloud

wordcloud_dir = os.path.join(repo_dir, 'wordclouds')
os.makedirs(wordcloud_dir, exist_ok=True)

before_freq = Counter(all_tokens)
after_freq = Counter([tok for sent in filtered_sentences for tok in sent])

for label, freq in [('before_stopword_removal', before_freq), ('after_stopword_removal', after_freq)]:
    wc = WordCloud(width=900, height=500, background_color='white', max_words=50)
    wc.generate_from_frequencies(freq)
    path = os.path.join(wordcloud_dir, f'{label}.png')
    wc.to_file(path)
    print(f'Saved {label}: {path}')

## Question 1.7 — Sentence probability

This cell computes sentence probability under different n-gram orders for example Amharic sentences.

In [ ]:
# 1.7 Sentence probabilities for n = 1, 2, 3, 4
sentence_examples = [
    ['ኢትዮጵያ', 'በአፍሪካ', 'ቀንድ', 'ትገኛለች'],
    ['ኢትዮጵያ', 'በአፍሪካ', 'ቀንድ', 'ትኖራለች'],
    ['ኢትዮጵያ', 'ውስጥ', 'የቡና', 'ምርት', 'በጣም', 'ከፍተኛ', 'ነው']
]

for n in [1, 2, 3, 4]:
    model = NGramLanguageModel(n=n, use_padding=False)
    model.train(tokenized_sentences)
    print(f'\n--- n={n} sentence probabilities ---')
    for s in sentence_examples:
        prob = model.sentence_probability(s, smoothing='laplace', alpha=0.1)
        print(f'{s} -> {prob:.6f}')

## Question 1.8 — Sentence generation

This cell samples and generates text from the trained n-gram models. It shows how the output becomes more specific as n increases.

In [ ]:
# 1.8 Random sentence generation using the full-corpus n-gram models
for n in [1, 2, 3, 4]:
    model = NGramLanguageModel(n=n, use_padding=False)
    model.train(tokenized_sentences)
    generated = model.generate_sentence(max_length=12, temperature=0.8, seed=42)
    print(f'\n--- n={n} generated sentence ---')
    print(' '.join(generated))

## Question 2: Intrinsic Evaluation

This section uses the cleaned full-corpus training data, splits it 80/20, and evaluates the n-gram models with perplexity.

In [ ]:
# Q2 uses the same preprocessing from Q1 and trains on an 80/20 split of the full corpus.
random.seed(42)
all_sentences = [sent for sent in tokenized_sentences if len(sent) >= 3]
random.shuffle(all_sentences)

split_idx = int(0.8 * len(all_sentences))
train_sentences = all_sentences[:split_idx]
test_sentences = all_sentences[split_idx:]

print(f'Train sentences: {len(train_sentences)}')
print(f'Test sentences: {len(test_sentences)}')

models = {}
for n in [1, 2, 3, 4]:
    model = NGramLanguageModel(n=n, use_padding=False)
    model.train(train_sentences)
    models[n] = model
    print(f'n={n}: vocab={len(model.vocab)}, ngrams={len(model.n_gram_counts)}')

eval_sample = test_sentences[:200]
for n in [1, 2, 3, 4]:
    ppl = models[n].compute_perplexity(eval_sample, smoothing='add_k', alpha=0.1)
    ce = math.log2(ppl) if ppl > 0 else float('inf')
    print(f'n={n}: cross_entropy={ce:.4f}, perplexity={ppl:.2f}')

for label, method, alpha in [
    ('Laplace', 'laplace', 1.0),
    ('Add-k (0.1)', 'add_k', 0.1),
    ('Add-k (0.01)', 'add_k', 0.01),
]:
    ppl = models[2].compute_perplexity(eval_sample, smoothing=method, alpha=alpha)
    print(f'Bigram | {label}: perplexity = {ppl:.2f}')

Train sentences: 15992
Test sentences: 3999
n=1: vocab=60140, ngrams=60140
n=2: vocab=60140, ngrams=158793
n=3: vocab=60140, ngrams=161568
n=4: vocab=60140, ngrams=148918
n=1: cross_entropy=14.4873, perplexity=22967.15
n=2: cross_entropy=14.5204, perplexity=23500.81
n=3: cross_entropy=15.1395, perplexity=36094.30
n=4: cross_entropy=15.2109, perplexity=37926.70
Bigram | Laplace: perplexity = 41153.96
Bigram | Add-k (0.1): perplexity = 23500.81
Bigram | Add-k (0.01): perplexity = 13384.34


## Question 3: Extrinsic Evaluation

This section tests whether the model can predict the next word in context and whether it prefers a coherent sentence over a corrupted one.

In [ ]:
# Q3 uses the same Q1 preprocessing pipeline and test/training split.
class BackoffPredictor:
    def __init__(self, models):
        self.models = models

    def get_candidates(self, context_tokens, top_k=5):
        for order in [4, 3, 2]:
            model = self.models[order]
            ctx = tuple(context_tokens[-(order - 1):]) if len(context_tokens) >= order - 1 else None
            if ctx and ctx in model.context_counts and model.context_counts[ctx]:
                return model.get_candidates(context_tokens, top_k=top_k)
        return self.models[2].get_candidates(context_tokens, top_k=top_k)

random.seed(42)
all_sentences_q3 = [sent for sent in tokenized_sentences if len(sent) >= 4]
random.shuffle(all_sentences_q3)

split_idx = int(0.85 * len(all_sentences_q3))
train_q3 = all_sentences_q3[:split_idx]
test_q3 = all_sentences_q3[split_idx:]

models_q3 = {}
for n in [1, 2, 3, 4]:
    model = NGramLanguageModel(n=n, use_padding=False)
    model.train(train_q3)
    models_q3[n] = model

predictor = BackoffPredictor(models_q3)
context = ['ኢትዮጵያ', 'ውስጥ', 'የቡና']
print('Context:', context)
print('Predicted next words:')
print(predictor.get_candidates(context, top_k=5))

eval_instances = []
for sentence in test_q3[:100]:
    for i in range(2, min(len(sentence) - 1, 6)):
        context = sentence[:i]
        true_target = sentence[i]
        eval_instances.append((context, true_target))
    if len(eval_instances) >= 200:
        break

top1 = 0
for context, true_word in eval_instances:
    candidates = predictor.get_candidates(context, top_k=10)
    candidate_words = [w for w, _ in candidates]
    if true_word in candidate_words:
        rank = candidate_words.index(true_word) + 1
        if rank == 1:
            top1 += 1

print(f'Evaluation instances: {len(eval_instances)}')
print(f'Top-1 accuracy: {(top1 / len(eval_instances)) * 100:.2f}%')

real_sentence = ['ኢትዮጵያ', 'በአፍሪካ', 'ቀንድ', 'ትገኛለች']
corrupted_sentence = ['ኢትዮጵያ', 'በአፍሪካ', 'ድንጋይ', 'ትገኛለች']
for n in [1, 2, 3, 4]:
    model = models_q3[n]
    real_logp = model.log_sentence_probability(real_sentence, smoothing='add_k', alpha=0.1)
    corr_logp = model.log_sentence_probability(corrupted_sentence, smoothing='add_k', alpha=0.1)
    print(f'n={n}: real={real_logp:.2f}, corrupted={corr_logp:.2f}')

Context: ['ኢትዮጵያ', 'ውስጥ', 'የቡና']
Predicted next words:
[('መሬት', 0.0001778956561115244), ('ሻጩን', 0.0001778956561115244), ('ቤት', 0.0001778956561115244), ('ዋጋ', 0.0001778956561115244)]
Evaluation instances: 200
Top-1 accuracy: 20.50%
n=1: real=-57.50, corrupted=-54.83
n=2: real=-39.94, corrupted=-43.40
n=3: real=-28.37, corrupted=-31.83
n=4: real=-15.92, corrupted=-15.92


## Question 4: Predictive Text System

This final section loads the predictive model and suggests likely next words for an Amharic query.

In [ ]:
predictor = AmharicPredictiveSystem()
predictor.train_or_load(corpus_path=CORPUS_PATH, max_sentences=None, force_retrain=False)

query = 'ኢትዮጵያ ውስጥ የቡና'
suggestions = predictor.predict_next_words(query, top_k=5, smoothing_alpha=0.1)
print(f'Query: {query}')
print('Top 5 next-word suggestions:')
for rank, (word, p) in enumerate(suggestions, 1):
    print(f'{rank}. {word} : {p:.6f}')

[*] Loading cached language models from /home/genesis/Desktop/NLP_assignment_presentation/NLP_AssignmentPRESENTATION/models/ngram_cache.pkl...
[✓] Language models loaded successfully from cache!
Query: ኢትዮጵያ ውስጥ የቡና
Top 5 next-word suggestions:
1. መገኛ : 0.384127
2. ቤት : 0.320635
3. ላይ : 0.098413
4. መረጃ : 0.098413
5. ጀበና : 0.098413
